# 01 Symbol Backtest - MA Cross

Backtest mot symbol cho chien luoc MA Cross bang source of truth trong `strategies.ma_cross`. Notebook nay khong tu viet lai signal: fast/slow MA, ATR, session mask va market execution deu di qua module chinh.

Muc tieu: kiem tra baseline theo M20/M30/M45, nhin trade log, equity, va canh bao sample size truoc khi toi uu.

In [ ]:
import sys
from pathlib import Path

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')

ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)

In [ ]:
from IPython.display import display

from core_python.strategies.ma_cross.params import SYMBOLS, TIMEFRAMES
from core_python.strategies.ma_cross.research_utils import (
    configure_notebook,
    run_timeframe_matrix,
    show_run_config,
    show_strategy_summary,
)
from core_python.strategies.ma_cross.symbol.backtest import run_symbol_backtest

configure_notebook()
show_strategy_summary()
print('Symbols:', ', '.join(SYMBOLS.keys()))
print('Timeframes:', ', '.join(TIMEFRAMES))

In [ ]:
RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'indicator_overrides': {},
    'strategy_overrides': {},
    'costs': {},
    'broker_profile': None,
}

show_run_config('MA Cross symbol backtest config', RUN_CONFIG)

In [ ]:
matrix, results = run_timeframe_matrix(**RUN_CONFIG)
display(matrix)

best_tf = matrix.sort_values(['sharpe', 'profit_factor'], ascending=False).iloc[0]['timeframe'] if not matrix.empty else 'M30'
result = results[best_tf]
print('Selected timeframe for inspection:', best_tf)
display(result.metrics)
display(result.trades.tail(20) if hasattr(result.trades, 'tail') else result.trades[-20:])
result.equity.plot(title=f"{RUN_CONFIG['symbol']} MA Cross equity - {best_tf}", figsize=(12, 4));